# Advanced Retrievers

## What are Advanced Retrievers?

Advanced retrievers in LlamaIndex are sophisticated components that go beyond simple vector similarity search to provide more nuanced, context-aware, and intelligent information retrieval. They combine multiple techniques such as:

- **Semantic Understanding**: Using embeddings to understand meaning and context
- **Keyword Matching**: Precise term-based search for exact specifications
- **Hierarchical Context**: Maintaining relationships between different levels of information
- **Multi-Query Processing**: Generating and combining results from multiple query variations
- **Fusion Techniques**: Intelligently combining results from different retrieval methods

## Why are Advanced Retrievers Important?

1. **Improved Accuracy**: Advanced retrievers can find more relevant information by using multiple search strategies
2. **Better Context Preservation**: They maintain important relationships between pieces of information
3. **Reduced Hallucination**: More precise retrieval leads to more accurate AI responses
4. **Scalability**: Efficient retrieval strategies work better with large document collections
5. **Flexibility**: Different retrieval methods can be combined for optimal results

## Index Types Overview

Before exploring advanced retrievers, it's helpful to first understand the three main index types supported by LlamaIndex. Each is designed to support different retrieval scenarios:

**VectorStoreIndex:**
- Stores vector embeddings for each document chunk
- Best suited for semantic retrieval based on meaning
- Commonly used in LLM pipelines and RAG applications

**DocumentSummaryIndex:**
- Generates and stores summaries of documents at indexing time
- Uses summaries to filter documents before retrieving full content
- Especially useful for large and diverse document sets that cannot fit in the context window of an LLM

**KeywordTableIndex:**
- Extracts keywords from documents and maps them to specific content chunks
- Enables exact keyword matching for rule-based or hybrid search scenarios
- Ideal for applications requiring precise term matching



In [1]:
import os
import json
from typing import List, Optional
import asyncio
import warnings
import numpy as np
warnings.filterwarnings('ignore')

# Core LlamaIndex imports
from llama_index.core import (
    VectorStoreIndex, 
    SimpleDirectoryReader, 
    Document,
    Settings,
    DocumentSummaryIndex,
    KeywordTableIndex
)
from llama_index.core.retrievers import (
    BaseRetriever,
    VectorIndexRetriever,
    AutoMergingRetriever,
    RecursiveRetriever,
    QueryFusionRetriever
)
# Advanced retriever imports
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.indices.document_summary import (
    DocumentSummaryIndexLLMRetriever,
    DocumentSummaryIndexEmbeddingRetriever,
)
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.core.schema import NodeWithScore, QueryBundle
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.embeddings import BaseEmbedding

# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# from sentence_transformers import SentenceTransformer
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Statistical libraries for fusion techniques
try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("⚠️ scipy not available - some advanced fusion features will be limited")

print("✅ All imports successful!")

✅ All imports successful!


In [ ]:
def create_openai_llm():
    """Create OpenAI LLM instance using official LlamaIndex integration."""
    try:
        llm = OpenAI(
            model="gpt-4o-mini",
            temperature=0.9,
            max_tokens=512,
        )
        print("✅ OpenAI LLM initialized using official LlamaIndex integration")
        return llm

    except Exception as e:
        print(f"⚠️ OpenAI initialization error: {e}")


In [9]:
embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

llm = create_openai_llm()

# Configure global settings
# llm and embed model are set into Settings. No need to set them for each index and retriever.
Settings.llm = llm
Settings.embed_model = embed_model
print("✅ OpenAI LLM and embeddings configured!")


✅ OpenAI LLM initialized using official LlamaIndex integration
✅ OpenAI LLM and embeddings configured!


In [8]:
# Sample data for the lab - AI/ML focused documents
SAMPLE_DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.",
    "Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.",
    "Natural language processing enables computers to understand, interpret, and generate human language.",
    "Computer vision allows machines to interpret and understand visual information from the world.",
    "Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.",
    "Supervised learning uses labeled training data to learn a mapping from inputs to outputs.",
    "Unsupervised learning finds hidden patterns in data without labeled examples.",
    "Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.",
    "Generative AI can create new content including text, images, code, and more.",
    "Large language models are trained on vast amounts of text data to understand and generate human-like text."
]

# Consistent query examples used throughout the lab
DEMO_QUERIES = {
    "basic": "What is machine learning?",
    "technical": "neural networks deep learning", 
    "learning_types": "different types of learning",
    "advanced": "How do neural networks work in deep learning?",
    "applications": "What are the applications of AI?",
    "comprehensive": "What are the main approaches to machine learning?",
    "specific": "supervised learning techniques"
}

print(f"📄 Loaded {len(SAMPLE_DOCUMENTS)} sample documents")
print(f"🔍 Prepared {len(DEMO_QUERIES)} consistent demo queries")
for i, doc in enumerate(SAMPLE_DOCUMENTS[:3], 1):
    print(f"{i}. {doc}")
print("...")

📄 Loaded 10 sample documents
🔍 Prepared 7 consistent demo queries
1. Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
2. Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
3. Natural language processing enables computers to understand, interpret, and generate human language.
...


## Initialize Lab Environment

In [10]:
class AdvancedRetrieversLab:
    def __init__(self):
        print("🚀 Initializing Advanced Retrievers Lab...")
        self.documents = [Document(text=text) for text in SAMPLE_DOCUMENTS]
        self.nodes = SentenceSplitter().get_nodes_from_documents(self.documents)
        
        print("📊 Creating indexes...")
        # Create various indexes.
        self.vector_index = VectorStoreIndex.from_documents(self.documents)
        self.document_summary_index = DocumentSummaryIndex.from_documents(self.documents)
        self.keyword_index = KeywordTableIndex.from_documents(self.documents)
        # Llama-index applies the default node parser (SentenceSplitter) and create its own nodes and Indexes those nodes.
        # Hence no, need to pass "self.nodes" into from_documents
        
        print("✅ Advanced Retrievers Lab Initialized!")
        print(f"📄 Loaded {len(self.documents)} documents")
        print(f"🔢 Created {len(self.nodes)} nodes")

# Initialize the lab
lab = AdvancedRetrieversLab()

🚀 Initializing Advanced Retrievers Lab...
📊 Creating indexes...
current doc id: e18dfd02-68d2-4179-90a3-90865afa39df
current doc id: 79f5ec9b-28ea-4226-963b-b32575ababb9
current doc id: 67945fe6-d5b6-4c36-b4e5-3f32d9e0a1b5
current doc id: 64bff71c-9a4b-42b9-a054-482e997ca43d
current doc id: 3993ad8d-7195-43e3-8795-d3fc5d1021b5
current doc id: 946ff8c4-fe47-45ed-81f9-23bd19d87ac7
current doc id: a392e1ea-100a-4326-b9a2-c29adf1105e5
current doc id: 71291542-27f7-43e7-9b0a-fce3608379e7
current doc id: 55fa4fd5-c80d-4b76-bf83-28ee2b4b7e0b
current doc id: ec41de10-14f4-4dcc-b256-27d6d26a14a5
✅ Advanced Retrievers Lab Initialized!
📄 Loaded 10 documents
🔢 Created 10 nodes


## 1. Vector Index Retriever - The Foundation

The `Vector Index Retriever` uses vector embeddings to find semantically related content, making it ideal for general-purpose search and widely used in retrieval-augmented generation (RAG) pipelines.

**How it works**: 
- Documents are split into nodes and embedded using the configured embedding model
- Query is converted to an embedding vector
- Returns nodes ranked by cosine similarity to the query embedding
- Generates embeddings in batches of 2048 nodes by default

**Strengths**: 
- Excellent semantic understanding and context awareness
- Handles synonyms and related concepts effectively
- Works well with natural language queries

**Limitations**: 
- May miss exact keyword matches when specific terms are crucial
- Requires a good embedding model for optimal performance
- Can be computationally intensive for large document collections

In [11]:
# Basic vector retriever
vector_retriever = VectorIndexRetriever(
    index=lab.vector_index,
    similarity_top_k=3
)

# Alternative creation method
alt_retriever = lab.vector_index.as_retriever(similarity_top_k=3)

query = DEMO_QUERIES["basic"]  # "What is machine learning?"
nodes = vector_retriever.retrieve(query)

print(f"Query: {query}")
print(f"Retrieved {len(nodes)} nodes:")
for i, node in enumerate(nodes, 1):
    print(f"{i}. Score: {node.score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    print()

Query: What is machine learning?
Retrieved 3 nodes:
1. Score: 0.6402
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Score: 0.5189
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

3. Score: 0.4660
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...



## 2. BM25 Retriever - Advanced Keyword-Based Search

BM25 is a keyword-based retrieval method that improves on TF-IDF by addressing some of its key limitations. It's widely used in production search systems including Elasticsearch and Apache Lucene.

`TF-IDF (Term Frequency-Inverse Document Frequency)`

### When to Use BM25

**Ideal for:**
- Technical documentation where exact terms matter
- Legal documents with specific terminology
- Product catalogs with precise specifications
- Academic papers with specialized vocabulary
- Applications requiring keyword-based retrieval rather than semantic similarity

**Advantages:**
- Excellent precision for exact term matches
- Fast computational performance
- Proven effectiveness in production systems
- No training required (unlike neural approaches)
- Interpretable scoring mechanism

**Limitations:**
- No semantic understanding (doesn't handle synonyms)
- Struggles with typos and variations
- Limited context understanding
- Requires careful parameter tuning for optimal performance


In [15]:
try:
    import Stemmer
    
    # Create BM25 retriever with default parameters
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=lab.nodes,
        similarity_top_k=3,
        stemmer=Stemmer.Stemmer("english"),
        language="english"
    )
    
    query = DEMO_QUERIES["technical"]  # "neural networks deep learning"
    nodes = bm25_retriever.retrieve(query)
    
    print(f"Query: {query}")
    print("BM25 analyzes exact keyword matches with sophisticated scoring")
    print(f"Retrieved {len(nodes)} nodes:")
    
    for i, node in enumerate(nodes, 1):
        score = node.score if hasattr(node, 'score') and node.score else 0
        print(f"{i}. BM25 Score: {score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        
        # Highlight which query terms appear in the text
        text_lower = node.text.lower()
        query_terms = query.lower().split()
        found_terms = [term for term in query_terms if term in text_lower]
        if found_terms:
            print(f"   → Found terms: {found_terms}")
        print()
        
except ImportError:
    print("⚠️ BM25Retriever requires 'pip install PyStemmer'")
  

Query: neural networks deep learning
BM25 analyzes exact keyword matches with sophisticated scoring
Retrieved 3 nodes:
1. BM25 Score: 2.5203
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...
   → Found terms: ['neural', 'networks', 'deep', 'learning']

2. BM25 Score: 0.3372
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...
   → Found terms: ['learning']

3. BM25 Score: 0.3024
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....
   → Found terms: ['learning']



## 3. Document Summary Index Retrievers

Document Summary Index Retrievers use document summaries instead of the actual documents to find relevant content, making them efficient for large collections. **They return the original documents, not their summaries.**

**How it works (from authoritative source)**:
- **Generates and stores summaries of documents** at indexing time
- **Uses summaries to filter documents** before retrieving full content
- **Two-stage Process**: First uses summaries to filter documents, then returns full document content
- **Especially useful for large, diverse corpora** that cannot fit in the context window of an LLM

**Two Retrieval Options**: 
1. **DocumentSummaryIndexLLMRetriever**: 
   - Uses a large language model to analyze the query against document summaries
   - Provides intelligent document selection but can be more time-consuming and expensive
   - Best for complex queries requiring nuanced understanding

2. **DocumentSummaryIndexEmbeddingRetriever**: 
   - Uses semantic similarity between the query and summary embeddings
   - Faster and more cost-effective than LLM-based approach
   - Good for straightforward similarity matching

In [17]:
# LLM-based document summary retriever
doc_summary_retriever_llm = DocumentSummaryIndexLLMRetriever(
    lab.document_summary_index,
    choice_top_k=3  # Number of documents to select. Default is 1.
)

# Embedding-based document summary retriever  
doc_summary_retriever_embedding = DocumentSummaryIndexEmbeddingRetriever(
    lab.document_summary_index,
    similarity_top_k=3  # Number of documents to select. Default is 1.
)

query = DEMO_QUERIES["learning_types"]  # "different types of learning"

print(f"Query: {query}")

print("\nA) LLM-based Document Summary Retriever:")
print("Uses LLM to select relevant documents based on summaries")
try:
    nodes_llm = doc_summary_retriever_llm.retrieve(query)
    print(f"Retrieved {len(nodes_llm)} nodes")
    for i, node in enumerate(nodes_llm[:2], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
        print(f"   Text: {node.text[:80]}...")
        print()
except Exception as e:
    print(f"LLM-based retrieval demo: {str(e)[:100]}...")

print("B) Embedding-based Document Summary Retriever:")
print("Uses vector similarity between query and document summaries")
try:
    nodes_emb = doc_summary_retriever_embedding.retrieve(query)
    print(f"Retrieved {len(nodes_emb)} nodes")
    for i, node in enumerate(nodes_emb[:2], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
        print(f"   Text: {node.text[:80]}...")
        print()
except Exception as e:
    print(f"Embedding-based retrieval demo: {str(e)[:100]}...")


Query: different types of learning

A) LLM-based Document Summary Retriever:
Uses LLM to select relevant documents based on summaries
Retrieved 3 nodes
1. Score: 8.0000
   Text: Machine learning is a subset of artificial intelligence that focuses on algorith...

2. Score: 7.0000
   Text: Reinforcement learning is a type of machine learning where agents learn to make ...

B) Embedding-based Document Summary Retriever:
Uses vector similarity between query and document summaries
Retrieved 3 nodes
1. (Document summary)
   Text: Reinforcement learning is a type of machine learning where agents learn to make ...

2. (Document summary)
   Text: Unsupervised learning finds hidden patterns in data without labeled examples....



## 4. Auto Merging Retriever - Hierarchical Context Preservation

Auto Merging Retriever is designed to preserve context in long documents using a hierarchical structure.\
**It uses hierarchical chunking to break documents into parent and child nodes, and if enough child nodes from the same parent are retrieved, the retriever returns the parent node instead.**

In it, Smaller child chunks are indexed in the vector store for precise matching, while larger parent chunks are stored in the docstore.

In [19]:
# Create hierarchical nodes
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[512, 256, 128]
)

hier_nodes = node_parser.get_nodes_from_documents(lab.documents)

# Create storage context with all nodes
from llama_index.core import StorageContext
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.vector_stores import SimpleVectorStore

docstore = SimpleDocumentStore()
docstore.add_documents(hier_nodes)

storage_context = StorageContext.from_defaults(docstore=docstore)

# Create base index
base_index = VectorStoreIndex(hier_nodes, storage_context=storage_context)
base_retriever = base_index.as_retriever(similarity_top_k=6)

# Create auto-merging retriever
auto_merging_retriever = AutoMergingRetriever(
    base_retriever, 
    storage_context,
    verbose=True
)

query = DEMO_QUERIES["advanced"]  # "How do neural networks work in deep learning?"
nodes = auto_merging_retriever.retrieve(query)

print(f"Query: {query}")
print(f"Auto-merged to {len(nodes)} nodes")
for i, node in enumerate(nodes[:3], 1):
    print(f"{i}. Score: {node.score:.4f}")
    print(f"   Text: {node.text[:120]}...")
    print()

> Merging 1 nodes into parent node.
> Parent node id: 958d639a-0ae0-4825-886f-30d7a2fef5ba.
> Parent node text: Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.

Query: How do neural networks work in deep learning?
Auto-merged to 2 nodes
1. Score: 0.5591
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in data....

2. Score: 0.3606
   Text: Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks....



## 5. Recursive Retriever - Multi-Level Reference Following


The Recursive Retriever is **designed to follow relationships between nodes using references**. **It can follow references from one node to another, such as citations in academic papers or other metadata links**, allowing it to **retrieve related content across documents or layers of abstraction**.

**How it works (from authoritative source)**:
- **Follows node references** - traverses relationships to find referenced content
- **Supports chunk and metadata linking** - handles different types of references
- **Multi-Level Navigation**: Can execute sub-queries on referenced retrievers or query engines
- **Network Building**: Creates a network of interconnected retrievers that can reference each other

**Reference Types Supported**:
1. **Chunk References**: Smaller child chunks refer to larger parent chunks for additional context
2. **Metadata References**: Summaries or generated questions refer to larger content chunks, such as citations in academic papers

**Configuration:**
- `retriever_dict`: Maps node IDs or keys to specific retrievers
- `query_engine_dict`: Maps keys to query engines for sub-queries
- Node metadata can contain references to other nodes or data structures

In [20]:
# Create documents with references
docs_with_refs = []
for i, doc in enumerate(lab.documents):
    # Add reference metadata
    ref_doc = Document(
        text=doc.text,
        metadata={
            "doc_id": f"doc_{i}",
            "references": [f"doc_{j}" for j in range(len(lab.documents)) if j != i][:2]
        }
    )
    docs_with_refs.append(ref_doc)

# Create index with referenced documents
ref_index = VectorStoreIndex.from_documents(docs_with_refs)

# Create retriever mapping
retriever_dict = {
    f"doc_{i}": ref_index.as_retriever(similarity_top_k=1)
    for i in range(len(docs_with_refs))
}

# Base retriever
base_retriever = ref_index.as_retriever(similarity_top_k=2)

# Add the root retriever to the dictionary
retriever_dict["vector"] = base_retriever

# Recursive retriever
recursive_retriever = RecursiveRetriever(
    "vector",
    retriever_dict=retriever_dict,
    query_engine_dict={},
    verbose=True
)

query = DEMO_QUERIES["applications"]  # "What are the applications of AI?"
try:
    nodes = recursive_retriever.retrieve(query)
    print(f"Query: {query}")
    print(f"Recursively retrieved {len(nodes)} nodes")
    for i, node in enumerate(nodes[:3], 1):
        print(f"{i}. Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
except Exception as e:
    print(f"Recursive retriever demo: {str(e)}")

Retrieving with query id None: What are the applications of AI?
Retrieving text node: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
Retrieving text node: Generative AI can create new content including text, images, code, and more.
Query: What are the applications of AI?
Recursively retrieved 2 nodes
1. Score: 0.4160
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Score: 0.3723
   Text: Generative AI can create new content including text, images, code, and more....



## 6. Query Fusion Retriever - Multi-Query Enhancement with Advanced Fusion

The Query Fusion Retriever **combines results from different retrievers** (such as vector-based and keyword-based methods) and **optionally generates multiple variations of a query using an LLM to improve coverage**. **The results are merged using fusion strategies** to improve recall.

**How it works (from authoritative source)**:
- **Combines results from multiple retrievers** - e.g., vector-based and keyword-based methods
- **Supports multiple query variations** - generates different formulations of the same query
- **Uses fusion strategies to improve recall** - sophisticated merging techniques
- **Improved Coverage**: Reduces impact of query formulation on final results

**Core capabilities**:
1. **Multiple Retriever Support**: Combines results from different retrievers
2. **Query Variation Generation**: Optionally generates multiple variations of a query using an LLM
3. **Fusion Strategies**: Merges results using sophisticated fusion techniques

**Fusion Strategies Supported (from authoritative source)**:
1. **Reciprocal Rank Fusion (RRF)**: **Combines rankings across queries** - robust and doesn't rely on score magnitudes
2. **Relative Score Fusion**: **Normalizes scores within each result set** - preserves the relative confidence of each retriever
3. **Distribution-Based Fusion**: **Uses statistical normalization** - ideal for handling score variability

**Configuration:**
- `num_queries`: Number of query variations to generate (default: 4)
- `mode`: Fusion strategy ("reciprocal_rerank", "relative_score", "dist_based_score")
- `similarity_top_k`: Number of results to retrieve per query
- `use_async`: Enable async processing for better performance

## 6.1 Reciprocal Rank Fusion (RRF) Mode

Reciprocal Rank Fusion is the most robust fusion method in QueryFusionRetriever, designed to combine ranked lists from multiple query variations by using the reciprocal of ranks, which reduces the impact of outliers and provides stable fusion results.

In [23]:
# Create QueryFusionRetriever with RRF mode
base_retriever = lab.vector_index.as_retriever(similarity_top_k=5)

query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

try:
    # Create query fusion retriever with RRF mode
    rrf_query_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="reciprocal_rerank",
        use_async=False,
        verbose=True
    )
    # "QueryFusionRetriever will:"
    # 1. Generate query variations using LLM
    # 2. Retrieve results for each variation
    # 3. Apply Reciprocal Rank Fusion
    
    print(f"\nQuery: {query}")

    nodes = rrf_query_fusion.retrieve(query)
    
    print(f"\nRRF Query Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Final RRF Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")


Query: What are the main approaches to machine learning?
Generated queries:
1. Different types of machine learning techniques and their applications
2. Overview of supervised, unsupervised, and reinforcement learning methods

RRF Query Fusion Results:
1. Final RRF Score: 0.0489
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

2. Final RRF Score: 0.0487
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

3. Final RRF Score: 0.0333
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...



In [28]:
# Combined vector-based and keyword-based retrievers
rrf_query_fusion = QueryFusionRetriever(
        [base_retriever, bm25_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="reciprocal_rerank",
        use_async=False,
        verbose=True
    )
nodes = rrf_query_fusion.retrieve(query)

for i, node in enumerate(nodes, 1):
        print(f"{i}. Final RRF Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()

Generated queries:
1. Key methodologies in machine learning
2. Overview of different machine learning techniques
1. Final RRF Score: 0.0989
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Final RRF Score: 0.0979
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

3. Final RRF Score: 0.0953
   Text: Computer vision allows machines to interpret and understand visual information from the world....



## 6.2 Relative Score Fusion Mode


Relative Score Fusion normalizes retrieval scores relative to the maximum score within each query variation's results, enabling effective combination when you want to preserve score magnitude information across different query formulations.

In [25]:
base_retriever = lab.vector_index.as_retriever(similarity_top_k=5)

query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

try:
    # Create query fusion retriever with relative score mode
    rel_score_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="relative_score",
        use_async=False,
        verbose=False
    )
    # QueryFusionRetriever with relative_score will:
    # 1. Generate query variations
    # 2. Normalize scores within each variation (score/max_score)
    # 3. Combine normalized scores
    
    print(f"\nQuery: {query}")
    
    nodes = rel_score_fusion.retrieve(query)
    
    print(f"\nRelative Score Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Combined Relative Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")


Query: What are the main approaches to machine learning?

Relative Score Fusion Results:
1. Combined Relative Score: 1.0000
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Combined Relative Score: 0.7137
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

3. Combined Relative Score: 0.1482
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...



## 6.3 Distribution-Based Score Fusion Mode

Distribution-Based Score Fusion uses statistical properties of score distributions from each query variation to normalize and combine retrieval results, providing the most sophisticated handling of score variability and reliability across different query formulations.

In [26]:
base_retriever = lab.vector_index.as_retriever(similarity_top_k=8)

query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

try:
    # Create query fusion retriever with distribution-based mode
    dist_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="dist_based_score",
        use_async=False,
        verbose=False
    )
    # QueryFusionRetriever with dist_based_score will:
    # 1. Generate query variations
    # 2. Analyze score distributions for each variation
    # 3. Apply statistical normalization (z-score, percentiles)
    # 4. Combine with distribution-aware weighting
    
    print(f"\nQuery: {query}")
    
    nodes = dist_fusion.retrieve(query)
    
    print(f"\nDistribution-Based Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Statistically Normalized Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")


Query: What are the main approaches to machine learning?

Distribution-Based Fusion Results:
1. Statistically Normalized Score: 0.8164
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Statistically Normalized Score: 0.7122
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

3. Statistically Normalized Score: 0.4731
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...



## Recommended Retrievers by Use Case

Based on the authoritative source and the characteristics of each retriever, here are recommended approaches for different scenarios:

**General Q&A Applications:**
- **Primary**: Vector Index Retriever for semantic understanding
- **Enhancement**: Combine with BM25 Retriever using Query Fusion for hybrid approach
- **Benefit**: Combines semantic relevance with keyword matching
- **From authoritative source**: "For general Q&A, use a vector index retriever, potentially combined with a BM25 retriever. This retriever fusion combines semantic relevance with keyword matching."

**Technical Documentation:**
- **Primary**: BM25 Retriever for exact term matching
- **Enhancement**: Vector Index Retriever as secondary for contextual flexibility
- **Benefit**: Prioritizes exact technical terms while maintaining semantic understanding
- **From authoritative source**: "For technical documents, especially those where exact terms need to be prioritized, consider making BM25 your primary retriever, with the vector index retriever adding contextual flexibility as a secondary retriever."

**Long Documents:**
- **Primary**: Auto Merging Retriever
- **Benefit**: Retrieves longer parent versions only if enough shorter child versions are retrieved, preserving context
- **From authoritative source**: "For long documents, the auto merging retriever is a great option, because it will retrieve longer parent versions only if enough shorter child versions are retrieved."

**Research Papers:**
- **Primary**: Recursive Retriever
- **Benefit**: Follows citations and references to retrieve relevant content from cited papers
- **From authoritative source**: "For research papers, use the recursive retriever in order to retrieve relevant content from cited papers."

**Large Document Collections:**
- **Primary**: Document Summary Index Retriever for initial filtering
- **Enhancement**: Followed by Vector Index Retriever for detailed search within relevant documents
- **Benefit**: Narrows down relevant documents first, then performs detailed retrieval
- **From authoritative source**: "For large document sets, consider using the document summary index retriever to narrow down the number of relevant documents, followed by a vector search within the remaining subset to retrieve the most pertinent content."
